In [1]:
import pandas as pd
from pathlib import Path

processed_path = Path("../../data/processed")

train_data = pd.read_parquet(processed_path / "ml_train.parquet")
test_data = pd.read_parquet(processed_path / "ml_test.parquet")

X_train = train_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_train = train_data["HOSPITAL_EXPIRE_FLAG"]
subject_id_train = train_data["SUBJECT_ID"]

X_test = test_data.drop(columns=["SUBJECT_ID", "HOSPITAL_EXPIRE_FLAG"])
y_test = test_data["HOSPITAL_EXPIRE_FLAG"]

In [2]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier

categorical_cols = [
    "gender",
    "admission_type",
    "admission_location"
]

numeric_cols = X_train.columns.difference(categorical_cols).tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", SimpleImputer(strategy="median"), numeric_cols),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ]
)

decision_tree_model = DecisionTreeClassifier(random_state=42)

decision_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", decision_tree_model)
    ]
)

In [3]:
from sklearn.model_selection import StratifiedGroupKFold, cross_validate

setup = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

decision_tree_results = cross_validate(
    decision_tree_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", decision_tree_results["test_roc_auc"].mean())
print("Precision:", decision_tree_results["test_precision"].mean())
print("Recall:", decision_tree_results["test_recall"].mean())
print("F1:", decision_tree_results["test_f1"].mean())

ROC-AUC: 0.6209837764516977
Precision: 0.3231567959788316
Recall: 0.3385691972927152
F1: 0.33061252113244527


In [4]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "model__max_depth": [3, 5, 10],
    "model__min_samples_split": [2, 10],
    "model__min_samples_leaf": [1, 5]
}

grid_search = GridSearchCV(
    estimator=decision_tree_pipeline,
    param_grid=param_grid,
    cv=setup,
    scoring="roc_auc",
    n_jobs=1
)

grid_search.fit(
    X_train,
    y_train,
    groups=subject_id_train
)

print("Best parameters:", grid_search.best_params_)
print("Best ROC-AUC:", grid_search.best_score_)

Best parameters: {'model__max_depth': 5, 'model__min_samples_leaf': 5, 'model__min_samples_split': 2}
Best ROC-AUC: 0.7602800819708444


In [5]:
best_tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=5,
    min_samples_split=2,
    random_state=42
)

best_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", best_tree_model)
    ]
)

best_tree_results = cross_validate(
    best_tree_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", best_tree_results["test_roc_auc"].mean())
print("Precision:", best_tree_results["test_precision"].mean())
print("Recall:", best_tree_results["test_recall"].mean())
print("F1:", best_tree_results["test_f1"].mean())

ROC-AUC: 0.7602800819708444
Precision: 0.5766774978511439
Recall: 0.14167509912026593
F1: 0.2260391126702951


In [6]:
balanced_tree_model = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=5,
    min_samples_split=2,
    class_weight="balanced",
    random_state=42
)

balanced_tree_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", balanced_tree_model)
    ]
)

balanced_tree_results = cross_validate(
    balanced_tree_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    scoring=["roc_auc", "precision", "recall", "f1"]
)

print("ROC-AUC:", balanced_tree_results["test_roc_auc"].mean())
print("Precision:", balanced_tree_results["test_precision"].mean())
print("Recall:", balanced_tree_results["test_recall"].mean())
print("F1:", balanced_tree_results["test_f1"].mean())

ROC-AUC: 0.7720350055831071
Precision: 0.2532261866039692
Recall: 0.7007160688301806
F1: 0.371031164012488


In [7]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

tree_probs = cross_val_predict(
    balanced_tree_pipeline,
    X_train,
    y_train,
    cv=setup,
    groups=subject_id_train,
    method="predict_proba"
)[:, 1]

thresholds = [0.5, 0.6, 0.7, 0.8]

for threshold in thresholds:
    y_pred = (tree_probs >= threshold).astype(int)

    print("Threshold:", threshold)
    print("Precision:", precision_score(y_train, y_pred))
    print("Recall:", recall_score(y_train, y_pred))
    print("F1:", f1_score(y_train, y_pred))
    print()

Threshold: 0.5
Precision: 0.251513894649523
Recall: 0.7007164317078808
F1: 0.3701623733365889

Threshold: 0.6
Precision: 0.3115018118373373
Recall: 0.5363993529003929
F1: 0.39412463915775175

Threshold: 0.7
Precision: 0.3557745954993491
Recall: 0.44210769586318466
F1: 0.3942704039571311

Threshold: 0.8
Precision: 0.4325518178729188
Recall: 0.29419921423619133
F1: 0.3502063273727648



## Decision Tree

A Decision Tree classifier was evaluated for ICU mortality prediction.

- Numerical features were processed using median imputation.
- Categorical features were transformed using `OneHotEncoder`.
- StandardScaler was not used because Decision Tree models are not sensitive to feature scaling.
- Model performance was evaluated using 5-fold `StratifiedGroupKFold`, keeping ICU stays from the same patient in the same fold.

### Baseline Decision Tree

The default Decision Tree showed weak performance:

- ROC-AUC: **0.621**
- Precision: **0.323**
- Recall: **0.339**
- F1: **0.331**

### Hyperparameter Tuning

`GridSearchCV` was used to test different values of:

- `max_depth`
- `min_samples_split`
- `min_samples_leaf`

The best parameters were:

- `max_depth = 5`
- `min_samples_leaf = 5`
- `min_samples_split = 2`

With these parameters:

- ROC-AUC: **0.760**
- Precision: **0.577**
- Recall: **0.142**
- F1: **0.226**

Although ROC-AUC improved substantially, recall became very low.

### Class Weight Balancing

Because mortality is the minority class, `class_weight="balanced"` was added to the tuned Decision Tree.

Performance improved to:

- ROC-AUC: **0.772**
- Precision: **0.253**
- Recall: **0.701**
- F1: **0.371**

Class weighting substantially increased recall, but precision decreased due to a higher number of false-positive predictions.

### Threshold Tuning

Different probability thresholds were evaluated for the tuned and balanced Decision Tree:

- Threshold `0.5`: Precision **0.252**, Recall **0.701**, F1 **0.370**
- Threshold `0.6`: Precision **0.312**, Recall **0.536**, F1 **0.394**
- Threshold `0.7`: Precision **0.356**, Recall **0.442**, F1 **0.394**
- Threshold `0.8`: Precision **0.433**, Recall **0.294**, F1 **0.350**

Increasing the threshold improved precision but reduced recall. The model was not able to achieve high precision and recall simultaneously.

Overall, the tuned and class-balanced Decision Tree provided much better recall than the baseline model, but its precision-recall balance remained limited.